# 11. 최종: 참조 태거 개선 (v3) → 최종 파이프라인
지금 파이프라인에서 아직 손대지 않은 부분이 **참조 모델인 v2 태거(0.9758)** 다. 08~10 은 '01 과 태거가 일치하는 어절을 고정'하고 태거의 답을 후보로 쓰므로, 태거가 좋아지면 고정과 후보가 함께 좋아진다.

| v3 에 넣는 것 | 출처 | 목적 |
|---|---|---|
| 생성 리뷰 10만 문장에 노이즈를 입혀 학습 | 10단계 | train 에 없던 어휘 |
| 실제 관찰된 난독화 형태로 어절 치환 | 깃허브 참고 | 가짜 노이즈가 아닌 실제 형태 |
| 입력 어절 일부를 정답으로 미리 복원 | 깃허브 참고 | 깨끗한 단어는 그대로 두기 |
| 새 샘플 이어 붙이기 | 깃허브 참고 | test 의 긴 문장 |
| 라벨 스무딩 0.1, 가중치 EMA 0.999 | 깃허브 참고 | 과신 완화, 안정화 |

(KoCharELECTRA 층 추가는 모델 구조 변경이 커서 제외)

| 비교 기준 | 검증 F1 |
|---|---|
| v2 태거 단독 | 0.9758 |
| 09 | 0.9908 |
| 10 (현재 최고) | 0.9912 |

**전제:** 10 노트북의 검증 단계 완료 (`outputs10/best_params.json`, `outputs/gen_reviews_split.txt`).
**GPU 런타임 (A100 권장), 첫 셀부터.**

## 0) 준비

In [ ]:
import os
os.chdir('/content')
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
DRIVE = '/content/drive/MyDrive/데이콘/난독화'
!mkdir -p /content/work && rsync -a --exclude '.playwright-mcp' --exclude '__pycache__' --exclude 'open.zip' --exclude 'v2_cache_*.pkl' "$DRIVE/" /content/work/
%cd /content/work
!ls outputs10/best_params.json outputs/gen_reviews_split.txt outputs/gen_reviews_full.txt outputs10/submission_noisy_channel.csv 2>&1; nproc; nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
def save_back():
    if not os.path.exists(DRIVE):
        drive.mount('/content/drive', force_remount=True)
    !rsync -a --update --exclude 'v2_cache_*.pkl' /content/work/outputs/ "$DRIVE/outputs/"
    !mkdir -p "$DRIVE/outputs10" && rsync -a --update /content/work/outputs10/ "$DRIVE/outputs10/"
    print('saved outputs, outputs10 -> drive')

## 1) 사전 특징 캐시 + 미세조정 언어모델 준비
사전 특징 캐시는 런타임마다 다시 만든다 (12코어 약 2분). 언어모델은 저장된 어댑터로 조립만 한다.

In [ ]:
!pip -q install peft
!pip -q uninstall -y torchao
!python -u lm_finetune.py --merge_only

## 2) v3 태거 학습 (30 epoch)
문장당 샘플: 실제 1 + pool 증강 1 + 새 샘플 4 (그중 50% 생성 리뷰, 15% 관찰 형태 치환, 나머지 채널 노이즈).
출력의 `v3 옵션:` 줄에서 설정을 확인. 결과는 `tagger_v2_best_v3.pt`, `val_pred_tagger_v2_v3.csv` (v2 결과를 덮어쓰지 않음).
epoch 별 기록은 `outputs/tagger_v2_runs.csv` (run_name = v3). **v2 단독 0.9758 과 비교.**

In [ ]:
!python -u tagger_v2.py --mode train --run_name v3 --epochs 30 --fresh 4 --gen_file outputs/gen_reviews_split.txt --gen_ratio 0.5 --subst_ratio 0.15 --partial 0.15 --concat 0.2 --label_smoothing 0.1 --ema 0.999 --loader_workers 8
save_back()

In [ ]:
import pandas as pd
r = pd.read_csv('outputs/tagger_v2_runs.csv')
g = r[r.run_name == 'v3'].dropna(subset=['val_f1'])
print('v3 best', g.val_f1.max(), 'epoch', int(g.loc[g.val_f1.idxmax(), 'epoch']), '| v2 scratch 0.9758')
g[['epoch', 'loss', 'val_f1', 'sec']].tail(8)

## 3) v3 태거의 test 예측 (최종 단계에서 참조로 사용)

In [ ]:
!python -u tagger_v2.py --mode predict --run_name v3 --workers 8
!cp outputs/val_pred_tagger_v2_v3.csv outputs/submission_tagger_v2_v3.csv outputs10/
save_back()

## 4) 최종 파이프라인 (검증): 증강한 01 + v3 태거 + 미세조정 언어모델
- `11` : 참조 = v3 만
- `11b`: 참조 = v2 와 v3 둘 다 (세 모델이 모두 일치해야 고정, 두 태거의 답이 모두 후보)

출력의 '고정 어절 정확도'와 '상한'을 10 (98.90%, 0.9958) 과 비교.

In [ ]:
!python -u anchor_infill.py --mode val --out_dir outputs10 --refs v3 --weights 0.9,1.0,1.1 --unanchor_oov --cand_beam 64 --nbest 20 --batch_tokens 8192 --lm models/qwen_ft_split --name 11
save_back()

In [ ]:
!python -u anchor_infill.py --mode val --out_dir outputs10 --refs v2,v3 --weights 0.9,1.0,1.1 --unanchor_oov --cand_beam 64 --nbest 20 --batch_tokens 8192 --lm models/qwen_ft_split --name 11b
save_back()

## 5) 결과 비교

In [ ]:
a = pd.read_csv('outputs10/anchor_runs.csv')
a['name'] = a['name'].astype(str)
print(a.sort_values('f1', ascending=False).groupby('name').head(1)[['name', 'refs', 'w', 'f1', 'base_f1', 'oracle_f1', 'sec']])
import glob
b10 = pd.read_csv('outputs10/val_pred_anchor_agree_v2_k20_b64_uo_10.csv').set_index('ID')
for f in sorted(glob.glob('outputs10/val_pred_anchor_*_11*.csv')):
    b = pd.read_csv(f).set_index('ID'); d = b.f1 - b10.f1
    print(f"{f.split('_')[-1][:-4]:4s} F1 {b.f1.mean():.4f} | 10 대비 좋아진 행 {(d>0).sum()} / 나빠진 행 {(d<0).sum()}")

## 6) test 예측 (5 에서 더 좋은 쪽으로)
10 노트북의 test 예측(5번 셀)이 끝나 있어야 한다: `outputs/gen_reviews_full.txt`, `outputs10/submission_noisy_channel.csv`.
아래는 `11` (참조 v3) 기준. `11b` 가 더 좋으면 `--refs v2,v3 --name 11b` 로 바꿀 것.
결과: `outputs10/submission_anchor_agree_v3_k20_b64_uo_11.csv`

In [ ]:
!cp "$DRIVE/outputs/gen_reviews_full.txt" outputs/
!cp "$DRIVE/outputs10/submission_noisy_channel.csv" "$DRIVE/outputs10/submission_noisy_channel_params.json" outputs10/


In [ ]:
!python -u lm_finetune.py --full --merge_only
!python -u anchor_infill.py --mode test --out_dir outputs10 --aug_file outputs/gen_reviews_full.txt --refs v2,v3 --unanchor_oov --cand_beam 64 --nbest 20 --batch_tokens 32768 --lm models/qwen_ft_full --name 11b
save_back()


### (선택) 런타임 해제

In [ ]:
from google.colab import runtime; runtime.unassign()